# EviChange — ground-truth labeling (Quang Ninh)

This notebook produces the **only ground truth** in the study: your visual judgement of
whether real change happened at ~30 sites, and what kind of change. The 3-condition comparison
is scored against these labels, so the labels must not depend on anything the system computes.

**Blind protocol (write this in the report's method section):**
1. Label **before** running any VLM condition on these sites, and never look at claims first.
2. You see only before/after image chips. You do **not** see index values, the detector's
   output, or whether a site came from the detector or from random sampling. Detected
   regions and random background sites are mixed and shuffled, with anonymous ids.
3. Every chip shows a yellow box; judge what happened **inside the box** (the box is the unit
   of ground truth). Random sites borrow the box size and shape of a real detected region, so box
   size and shape carry no clue about where a site came from.
4. `unsure` is a valid answer. Don't guess to fill the form.
5. After ≥7 days, relabel 10 random sites (Step 7) to measure your own consistency.

**Before you start:** run the Phase 2 Earth Engine extraction so
`change_regions_quangninh_v1.json` exists in Drive. Allow about 1 hour for 30 sites.

Run cells top to bottom with Shift+Enter.

## Step 1 — Connect Earth Engine and Drive

In [ ]:
!pip -q install earthengine-api
import ee, json, os, random, time, datetime, urllib.request
from google.colab import drive
drive.mount('/content/drive')

EE_PROJECT = 'YOUR_PROJECT_ID'          # the Cloud project registered for Earth Engine
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

ROOT  = '/content/drive/MyDrive/evichange/labeling'
CHIPS = f'{ROOT}/chips'
os.makedirs(CHIPS, exist_ok=True)
print('Earth Engine + Drive ready:', ROOT)

## Step 2 — Study settings
Both windows are **dry season** (Nov–Mar), so seasonal vegetation differences are not mistaken for change.
They **must be the same windows the Phase 2 extraction used**; Step 4 stops if the regions file says otherwise.
`REGIONS_FILE` is the Phase 2 output. `N_DETECTED` + `N_RANDOM` = sites to label.

In [ ]:
AOI_BOUNDS = [107.22, 20.98, 107.40, 21.10]      # [west, south, east, north]  Cam Pha, Quang Ninh
BEFORE = ('2018-11-01', '2019-03-31')
AFTER  = ('2022-11-01', '2023-03-31')
REGIONS_FILE = '/content/drive/MyDrive/evichange/outputs/change_regions_quangninh_v1.json'

N_DETECTED  = 20       # detected regions to label (spread across the size range)
N_RANDOM    = 10       # random background sites outside every detected region
SEED        = 20260926
CHIP_MIN_M  = 600      # smallest chip side, metres
CHIP_PX     = 512
PROTOCOL    = 'labels_v1'
LABELER     = 'labeler1'      # your initials; a second labeler uses their own

## Step 3 — Cloud-masked dry-season composites
Uses the **SCL** scene-classification band for cloud masking. `QA60` cloud bits are not populated for
much of 2022–2024 in `S2_SR_HARMONIZED`, so a QA60 mask silently leaves clouds in the "after" image.
You should see two scene counts greater than 0.

In [ ]:
S2 = 'COPERNICUS/S2_SR_HARMONIZED'
BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
aoi = ee.Geometry.Rectangle(AOI_BOUNDS)

def mask_scl(img):
    scl = img.select('SCL')
    clear = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))      # cloud shadow, cloud medium/high
             .And(scl.neq(10)).And(scl.neq(11)))              # cirrus, snow
    return img.select(BANDS).updateMask(clear).divide(10000)

def composite(window):
    col = (ee.ImageCollection(S2).filterBounds(aoi).filterDate(*window)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60)).map(mask_scl))
    return col.median().clip(aoi), col.size().getInfo()

before_img, n_before = composite(BEFORE)
after_img,  n_after  = composite(AFTER)
print('before scenes:', n_before, ' after scenes:', n_after)
assert n_before > 0 and n_after > 0, 'widen the windows'

## Step 4 — Choose the sites (detected + random), shuffle, anonymise
Detected regions are sampled across the size range, so it isn't just the largest blobs.
Random sites are points at least 200 m from any detected region. Each one borrows the box size and shape
of a randomly chosen detected region, and is redrawn if that box would touch a detected region.
The mapping from anonymous id to origin goes into `sites_key.json`. **Don't open that file until Step 8.**

In [ ]:
import math
rng = random.Random(SEED)
fc = json.load(open(REGIONS_FILE))
meta = fc.get('metadata', {})
if meta:   # written by agents/ee_engine.py (Phase 2)
    assert tuple(meta['before_window']) == BEFORE and tuple(meta['after_window']) == AFTER, \
        f"windows differ from the detector's: {meta['before_window']} / {meta['after_window']}"
else:
    print('WARNING: regions file has no metadata; check by hand that BEFORE/AFTER match the detector run')
all_feats = fc['features']
feats = sorted(all_feats, key=lambda f: f['properties']['area_ha'])
assert len(feats) >= N_DETECTED, f'only {len(feats)} detected regions; the extraction must not cap the count'
step = len(feats) / N_DETECTED                     # even spread over the size ranking
feats = [feats[int(i * step)] for i in range(N_DETECTED)]
print(f'{len(feats)} of {len(all_feats)} detected regions selected')

detected_raw = ee.FeatureCollection([ee.Feature(ee.Geometry(f['geometry'])) for f in all_feats]).geometry()

def bbox_of(geom):
    ring = ee.Geometry(geom).bounds(1).coordinates().getInfo()[0]
    xs, ys = [p[0] for p in ring], [p[1] for p in ring]
    return min(xs), min(ys), max(xs), max(ys)

def rect(cx, cy, w_deg, h_deg):
    return ee.Geometry.Rectangle([cx - w_deg / 2, cy - h_deg / 2, cx + w_deg / 2, cy + h_deg / 2])

def chip_side_m(w_deg, h_deg, lat):
    m_per_deg = 111_320
    return max(CHIP_MIN_M, 1.6 * max(w_deg * m_per_deg * math.cos(math.radians(lat)), h_deg * m_per_deg))

# outline = the region's bounding box; chip = a square around it. Box sizes of ALL detected regions
# are the pool random sites draw from, so size/shape don't reveal origin.
boxes = [bbox_of(f['geometry']) for f in all_feats]
sites = []
for f in feats:
    w, s, e, n = bbox_of(f['geometry'])
    cx, cy = (w + e) / 2, (s + n) / 2
    side = chip_side_m(e - w, n - s, cy)
    sites.append({'origin': 'detected', 'region_id': f['properties']['id'], 'lon': cx, 'lat': cy,
                  'outline': rect(cx, cy, e - w, n - s).getInfo(),
                  'chip': ee.Geometry.Point([cx, cy]).buffer(side / 2).bounds(1).getInfo()})

excl = aoi.difference(detected_raw.buffer(200), 1)
cands = ee.FeatureCollection.randomPoints(excl, N_RANDOM * 5, SEED).getInfo()['features']
for p in cands:
    if sum(s['origin'] == 'random' for s in sites) == N_RANDOM:
        break
    cx, cy = p['geometry']['coordinates']
    w, s, e, n = rng.choice(boxes)
    box = rect(cx, cy, e - w, n - s)
    if not aoi.contains(box, 1).getInfo() or box.intersects(detected_raw, 1).getInfo():
        continue
    side = chip_side_m(e - w, n - s, cy)
    sites.append({'origin': 'random', 'region_id': None, 'lon': cx, 'lat': cy, 'outline': box.getInfo(),
                  'chip': ee.Geometry.Point([cx, cy]).buffer(side / 2).bounds(1).getInfo()})
n_rand = sum(s['origin'] == 'random' for s in sites)
print(f'{n_rand} random background sites selected')
assert n_rand == N_RANDOM, 'not enough clean background points: raise the candidate count'

rng.shuffle(sites)
for i, s in enumerate(sites, 1):
    s['site_id'] = f's{i:02d}'
json.dump({s['site_id']: {'origin': s['origin'], 'region_id': s['region_id']} for s in sites},
          open(f'{ROOT}/sites_key.json', 'w'), indent=1)
public = [{k: s[k] for k in ('site_id', 'lon', 'lat', 'outline', 'chip')} for s in sites]
json.dump(public, open(f'{ROOT}/sites.json', 'w'), indent=1)
print(len(public), 'sites saved (origin hidden in sites_key.json)')

## Step 5 — Render the chips
Four PNGs per site: true colour and SWIR false colour (B12-B8-B4; bare soil and spoil show up clearly),
before and after. **Both dates use identical stretch settings**, so colour differences are real.

In [ ]:
VIS = {'rgb': (['B4', 'B3', 'B2'], 0.0, 0.3), 'swir': (['B12', 'B8', 'B4'], 0.0, 0.45)}

def chip(img, kind, site):
    bands, lo, hi = VIS[kind]
    edge = (ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(ee.Geometry(site['outline']))]), 1, 2)
            .visualize(palette=['ffff00']))
    url = img.visualize(bands=bands, min=lo, max=hi).blend(edge).getThumbURL(
        {'region': site['chip'], 'dimensions': CHIP_PX, 'format': 'png'})
    return urllib.request.urlopen(url, timeout=120).read()

for s in public:
    for when, img in (('before', before_img), ('after', after_img)):
        for kind in VIS:
            path = f"{CHIPS}/{s['site_id']}_{when}_{kind}.png"
            if not os.path.exists(path):
                open(path, 'wb').write(chip(img, kind, s))
    print('chips ready:', s['site_id'])

## Step 6 — Label (the form)
One site at a time. Progress is saved to Drive after every **Save**, so you can stop and resume.
- **Did real land-cover change happen inside the box?** yes / no / unsure
- **Change types:** tick all that apply (only if *yes*)
- **Mining-related?** Visible pit, benches/terraces, spoil dump, haul road, coal stockpile. This is what you
  *see*; legality is never labelled because it can't be seen. The Google Maps link shows **today's** imagery,
  which is later than the after window. Use it only to recognise surface types, never to decide whether change
  happened by the after date (Google Earth Pro's historical-imagery slider can show the right year). Tick *used high-res*.
- **Confidence** in your own label.

In [ ]:
import ipywidgets as W
from IPython.display import display

LABELS_FILE = f'{ROOT}/{PROTOCOL}.jsonl'
CHANGE_TYPES = [('vegetation loss', 'vegetation_loss'), ('vegetation gain', 'vegetation_gain'),
                ('more bare/built surface', 'built_up_increase'), ('less bare/built surface', 'built_up_decrease'),
                ('more water', 'water_increase'), ('less water', 'water_decrease'), ('other', 'other')]

def labeled_ids(path):
    return {json.loads(l)['site_id'] for l in open(path)} if os.path.exists(path) else set()

def run_form(site_list, out_path):
    todo = [s for s in site_list if s['site_id'] not in labeled_ids(out_path)]
    box = W.VBox(); display(box)
    state = {'i': 0, 't0': time.time()}

    def show():
        if state['i'] >= len(todo):
            box.children = [W.HTML(f'<b>All {len(site_list)} sites labelled.</b> Saved to {out_path}')]
            return
        s = todo[state['i']]; sid = s['site_id']; state['t0'] = time.time()
        def img(when, kind):
            data = open(f'{CHIPS}/{sid}_{when}_{kind}.png', 'rb').read()
            return W.VBox([W.HTML(f'<b>{when}</b> ({kind})'), W.Image(value=data, width=360)])
        change = W.RadioButtons(options=['yes', 'no', 'unsure'], value=None, description='Change?')
        types = [W.Checkbox(description=lbl, value=False, indent=False) for lbl, _ in CHANGE_TYPES]
        mining = W.RadioButtons(options=['yes', 'no', 'unsure'], value=None, description='Mining?')
        conf = W.ToggleButtons(options=['low', 'medium', 'high'], value=None, description='Confidence')
        hires = W.Checkbox(description='used high-res imagery', value=False, indent=False)
        notes = W.Text(placeholder='notes (required if "other" is ticked)', layout=W.Layout(width='600px'))
        msg = W.HTML()
        save = W.Button(description='Save & next', button_style='success')
        link = f"https://www.google.com/maps/@{s['lat']:.5f},{s['lon']:.5f},800m/data=!3m1!1e3"

        def on_save(_):
            picked = [code for (lbl, code), cb in zip(CHANGE_TYPES, types) if cb.value]
            problems = []
            if change.value is None: problems.append('answer "Change?"')
            if mining.value is None: problems.append('answer "Mining?"')
            if conf.value is None: problems.append('pick a confidence')
            if change.value == 'no' and picked: problems.append('no change but change types ticked')
            if change.value == 'yes' and not picked: problems.append('tick at least one change type')
            if 'other' in picked and not notes.value.strip(): problems.append('describe "other" in notes')
            if problems:
                msg.value = '<span style="color:#b00">' + '; '.join(problems) + '</span>'
                return
            rec = {'site_id': sid, 'protocol': PROTOCOL, 'labeler': LABELER,
                   'labeled_at': datetime.datetime.now().isoformat(timespec='seconds'),
                   'change_occurred': change.value, 'change_types': picked, 'mining_related': mining.value,
                   'confidence': conf.value, 'used_highres': hires.value, 'notes': notes.value.strip(),
                   'seconds_spent': round(time.time() - state['t0']),
                   'before_window': list(BEFORE), 'after_window': list(AFTER)}
            with open(out_path, 'a') as fh:
                fh.write(json.dumps(rec) + '\n')
            state['i'] += 1
            show()

        save.on_click(on_save)
        box.children = [
            W.HTML(f'<h3>Site {sid} — {state["i"] + 1} of {len(todo)}</h3>'
                   f'<a href="{link}" target="_blank">open high-res (Google Maps, TODAY&#39;s imagery, not the after date)</a>'),
            W.HBox([img('before', 'rgb'), img('after', 'rgb')]),
            W.HBox([img('before', 'swir'), img('after', 'swir')]),
            W.HBox([change, mining]), W.HTML('<b>Change types</b> (tick all that apply)'),
            W.HBox(types[:4]), W.HBox(types[4:]), conf, hires, notes, save, msg]
    show()

sites_public = json.load(open(f'{ROOT}/sites.json'))
run_form(sites_public, LABELS_FILE)

## Step 7 — Self-consistency relabel (run ≥7 days after Step 6)
Relabels 10 random sites **without showing your earlier answers**. Step 8 compares them (Cohen's κ).
If a second person labels, set `LABELER` to their initials, and point `run_form` at
`labels_v1_<initials>.jsonl` with *all* sites. That gives inter-rater κ, which is stronger than self-agreement.

In [ ]:
RELABEL_FILE = f'{ROOT}/{PROTOCOL}_relabel.jsonl'
relabel_sites = random.Random(SEED + 1).sample(sites_public, k=min(10, len(sites_public)))
run_form(relabel_sites, RELABEL_FILE)

## Step 8 — Unblind and summarise (only after ALL labelling is finished)
Reports:
- **Detector precision**: labelled-as-change share of the *detected* sites.
- **Miss rate estimate**: labelled-as-change share of the *random* sites.
- **Agreement**: Cohen's κ on `change_occurred`, and per-type agreement.

With ~30 sites, every proportion is shown with a Wilson 95% interval, not just a point value.

In [ ]:
from math import sqrt

def load(path):
    return {r['site_id']: r for r in map(json.loads, open(path))} if os.path.exists(path) else {}

def wilson(k, n, z=1.96):
    if n == 0:
        return (float('nan'),) * 3
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)

def cohens_kappa(a, b):
    cats = sorted(set(a) | set(b)); n = len(a)
    if n == 0:
        return float('nan')
    po = sum(x == y for x, y in zip(a, b)) / n
    pe = sum((a.count(c) / n) * (b.count(c) / n) for c in cats)
    return 1.0 if pe == 1 else (po - pe) / (1 - pe)

labels, key = load(LABELS_FILE), json.load(open(f'{ROOT}/sites_key.json'))
for origin in ('detected', 'random'):
    rows = [labels[s] for s, v in key.items() if v['origin'] == origin and s in labels]
    sure = [r for r in rows if r['change_occurred'] != 'unsure']
    k = sum(r['change_occurred'] == 'yes' for r in sure)
    p, lo, hi = wilson(k, len(sure))
    print(f'{origin:9s}: {k}/{len(sure)} labelled change ({p:.0%}, 95% CI {lo:.0%}-{hi:.0%}); '
          f'{len(rows) - len(sure)} unsure')

relabels = load(RELABEL_FILE)
both = sorted(set(labels) & set(relabels))
if both:
    a = [labels[s]['change_occurred'] for s in both]; b = [relabels[s]['change_occurred'] for s in both]
    print(f'self-agreement on change_occurred: kappa={cohens_kappa(a, b):.2f} over {len(both)} sites')
    for _, t in CHANGE_TYPES:
        a = [t in labels[s]['change_types'] for s in both]; b = [t in relabels[s]['change_types'] for s in both]
        if any(a) or any(b):
            print(f'  {t:18s} agreement {sum(x == y for x, y in zip(a, b))}/{len(both)}')

## Step 9 — Bring the labels into the repo
Download `labels_v1.jsonl`, `labels_v1_relabel.jsonl`, `sites.json` and `sites_key.json` from
`MyDrive/evichange/labeling/` into `data/quangninh/labels/` in the repo. The evaluation code reads them
from there. **Never edit a label after Phase 4 has produced claims for that site.** If a label is
wrong, put the correction in a new `labels_v2.jsonl` with the reason in `notes`, so the change is visible.